# Using Captum
Captum is a model interpretability and understanding library for PyTorch. Captum means comprehension in Latin and contains general purpose implementations of integrated gradients, saliency maps, smoothgrad, vargrad and others for PyTorch models. It has quick integration for models built with domain-specific libraries such as torchvision, torchtext, and others.

In [19]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import huggingface_hub
import torch

from huggingface_hub import hf_hub_download
from huggingface_hub import login
from transformers import pipeline

from captum.attr import LayerIntegratedGradients

## load dataframes which we investigate

In [5]:
df_bear_but_pred_bull = pd.read_csv("../data/predictions/df_bear_but_pred_bull.csv")
df_bull_but_pred_bear = pd.read_csv("../data/predictions/df_bull_but_pred_bear.csv")

In [7]:
df_bear_but_pred_bull.head(10)

,text,label,predicted_scores,predicted_label,negative,neutral,positive,pred_top_score
0,Grubhub gains a bear on margin view,negative,"{'positive': 0.8692758679389954, 'negative': 0...",positive,0.068459,0.062265,0.869276,0.869276
1,Needham's Martin Defends Her Prediction That N...,negative,"{'positive': 0.5826396346092224, 'neutral': 0....",positive,0.092902,0.324459,0.582640,0.582640
2,Rio Tinto downgraded to neutral from overweigh...,negative,"{'positive': 0.5796186327934265, 'negative': 0...",positive,0.294971,0.125410,0.579619,0.579619
3,Royal Caribbean stock price target cut to $100...,negative,"{'positive': 0.44598066806793213, 'negative': ...",positive,0.434099,0.119920,0.445981,0.445981
4,SunTrust latest to trim Nielsen target,negative,"{'positive': 0.7235925197601318, 'neutral': 0....",positive,0.077757,0.198651,0.723593,0.723593
5,Nike to record $425 mln charge as it transitio...,negative,"{'positive': 0.7930759191513062, 'neutral': 0....",positive,0.019527,0.187397,0.793076,0.793076
6,Oracle’s Need for ‘Steep’ Sales Climb Leaves S...,negative,"{'positive': 0.6714838743209839, 'negative': 0...",positive,0.235818,0.092698,0.671484,0.671484
7,TikTok looks to diversify users as U.S. pressu...,negative,"{'positive': 0.9266785979270935, 'neutral': 0....",positive,0.008610,0.064711,0.926679,0.926679
8,Trading on the News Adds a Risk Premium to Sha...,negative,"{'positive': 0.9199829697608948, 'neutral': 0....",positive,0.025549,0.054468,0.919983,0.919983
9,Trans Mountain expansion costs soars to C$12.6...,negative,"{'positive': 0.9197039604187012, 'negative': 0...",positive,0.049697,0.030599,0.919704,0.919704


## retrieve Finbert model

In [8]:
login()

In [21]:
# get classifier
classifier = pipeline("text-classification", model="ProsusAI/finbert")
model = classifier.model
tokenizer = classifier.tokenizer

model.eval() # evaluation mode

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5603.44it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [12]:
# for a whole batch
def predict_batch(texts):
    preds = classifier(texts, batch_size=512, truncation=True, max_length=512, top_k=None)
    rows = []
    for row in preds:
        rows.append({d["label"]: float(d["score"]) for d in row})
    return rows

# example with dataframe texts
texts = df_bear_but_pred_bull["text"].astype(str).tolist()
predictions = predict_batch(texts)
predictions

[{'positive': 0.8692759871482849,
  'negative': 0.06845902651548386,
  'neutral': 0.06226496398448944},
 {'positive': 0.5826385021209717,
  'neutral': 0.3244595527648926,
  'negative': 0.09290196746587753},
 {'positive': 0.5796201825141907,
  'negative': 0.2949697971343994,
  'neutral': 0.12540997564792633},
 {'positive': 0.44598087668418884,
  'negative': 0.434099018573761,
  'neutral': 0.11992007493972778},
 {'positive': 0.7235931158065796,
  'neutral': 0.19865047931671143,
  'negative': 0.07775641977787018},
 {'positive': 0.7930759191513062,
  'neutral': 0.1873966008424759,
  'negative': 0.01952746883034706},
 {'positive': 0.67148357629776,
  'negative': 0.23581823706626892,
  'neutral': 0.09269817173480988},
 {'positive': 0.9266787171363831,
  'neutral': 0.06471139192581177,
  'negative': 0.008609970100224018},
 {'positive': 0.9199829697608948,
  'neutral': 0.05446789041161537,
  'negative': 0.025549182668328285},
 {'positive': 0.9197039604187012,
  'negative': 0.049696631729602814

In [ ]:
# Define forward function
def forward_func(input_ids, attention_mask):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    return outputs.logits

# Access FinBERT's word embedding layer
embedding_layer = model.base_model.embeddings.word_embeddings

lig = LayerIntegratedGradients(
    forward_func,
    embedding_layer
)

def explain_tweet(tweet, n_steps=50):

    device = next(model.parameters()).device

    inputs = tokenizer(
        tweet,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Get prediction
    with torch.no_grad():
        logits = forward_func(input_ids, attention_mask)
        probabilities = torch.softmax(logits, dim=1)

    predicted_class = logits.argmax(dim=1).item()
    predicted_label = model.config.id2label[predicted_class]

    # Baseline: replace normal tokens with padding
    baseline_ids = torch.full_like(
        input_ids,
        tokenizer.pad_token_id
    )

    # Preserve special tokens
    special_tokens = tokenizer.get_special_tokens_mask(
        input_ids[0].tolist(),
        already_has_special_tokens=True
    )

    for i, is_special in enumerate(special_tokens):
        if is_special:
            baseline_ids[0, i] = input_ids[0, i]

    # Calculate attributions
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask,),
        target=predicted_class,
        n_steps=n_steps,
        return_convergence_delta=True
    )

    # Aggregate embedding dimensions
    scores = attributions.sum(dim=-1).squeeze(0)

    tokens = tokenizer.convert_ids_to_tokens(
        input_ids[0]
    )

    print("Tweet:", tweet)
    print("Prediction:", predicted_label)
    print("Confidence:", probabilities[0, predicted_class].item())
    print("\nToken contributions:")

    for token, score in zip(tokens, scores):
        print(f"{token:20s} {score.item():+.5f}")

    print("\nConvergence delta:", delta.item())

    return {
        "tweet": tweet,
        "prediction": predicted_label,
        "confidence": probabilities[0].tolist(),
        "tokens": tokens,
        "attributions": scores.detach().cpu().tolist()
    }


# Test on a misclassified tweet
result = explain_tweet(
    df_bear_but_pred_bull["text"].iloc[9]
)

Tweet: Trans Mountain expansion costs soars to C$12.6B - report
Prediction: positive
Confidence: 0.9197039008140564

Token contributions:
[CLS]                +0.00000
trans                -0.03988
mountain             +0.02905
expansion            +0.48567
costs                +0.20672
so                   +0.11384
##ars                +0.21266
to                   +0.20806
c                    +0.06054
$                    +0.34086
12                   -0.00040
.                    +0.02922
6                    +0.33847
##b                  +0.09801
-                    +0.19800
report               -0.21585
[SEP]                +0.00000

Convergence delta: -0.13283610343933105


{'tweet': 'Trans Mountain expansion costs soars to C$12.6B - report',
 'prediction': 'positive',
 'confidence': [0.9197039008140564,
  0.049696702510118484,
  0.030599400401115417],
 'tokens': ['[CLS]',
  'trans',
  'mountain',
  'expansion',
  'costs',
  'so',
  '##ars',
  'to',
  'c',
  '$',
  '12',
  '.',
  '6',
  '##b',
  '-',
  'report',
  '[SEP]'],
 'attributions': [0.0,
  -0.039875537157058716,
  0.029054762795567513,
  0.48566919565200806,
  0.20671793818473816,
  0.11383862048387527,
  0.21266472339630127,
  0.20805734395980835,
  0.06053803116083145,
  0.3408642113208771,
  -0.0004042908549308777,
  0.0292232483625412,
  0.33846810460090637,
  0.0980079248547554,
  0.19799861311912537,
  -0.21584990620613098,
  0.0]}

In [25]:
print(df_bear_but_pred_bull.iloc[9])
result

text                Trans Mountain expansion costs soars to C$12.6...
label                                                        negative
predicted_scores    {'positive': 0.9197039604187012, 'negative': 0...
predicted_label                                              positive
negative                                                     0.049697
neutral                                                      0.030599
positive                                                     0.919704
pred_top_score                                               0.919704
Name: 9, dtype: object


{'tweet': 'Trans Mountain expansion costs soars to C$12.6B - report',
 'prediction': 'positive',
 'confidence': [0.9197039008140564,
  0.049696702510118484,
  0.030599400401115417],
 'tokens': ['[CLS]',
  'trans',
  'mountain',
  'expansion',
  'costs',
  'so',
  '##ars',
  'to',
  'c',
  '$',
  '12',
  '.',
  '6',
  '##b',
  '-',
  'report',
  '[SEP]'],
 'attributions': [0.0,
  -0.039875537157058716,
  0.029054762795567513,
  0.48566919565200806,
  0.20671793818473816,
  0.11383862048387527,
  0.21266472339630127,
  0.20805734395980835,
  0.06053803116083145,
  0.3408642113208771,
  -0.0004042908549308777,
  0.0292232483625412,
  0.33846810460090637,
  0.0980079248547554,
  0.19799861311912537,
  -0.21584990620613098,
  0.0]}

In [17]:
df_bear_but_pred_bull["text"].iloc[2]

'Rio Tinto downgraded to neutral from overweight at J.P. Morgan'

## visualize tweets

In [26]:
from html import escape

def visualize_tweet(result):

    tokens = result["tokens"]
    scores = result["attributions"]

    # Normalize scores for visualization
    max_score = max(abs(s) for s in scores) or 1

    html_tokens = []

    for token, score in zip(tokens, scores):

        # Remove WordPiece notation
        word = token.replace("##", "")

        intensity = abs(score) / max_score

        if score > 0:
            color = f"rgba(0, 180, 0, {intensity:.2f})"
        else:
            color = f"rgba(220, 0, 0, {intensity:.2f})"

        html_tokens.append(
            f'<span title="Attribution: {score:.5f}" '
            f'style="background-color:{color}; '
            f'padding:4px; margin:2px; '
            f'border-radius:4px;">'
            f'{escape(word)}</span>'
        )

    highlighted_tweet = " ".join(html_tokens)

    html_content = f"""
    <html>
    <body style="font-family:Arial; padding:25px;">

        <h2>FinBERT Prediction</h2>

        <p>
            <b>Sentiment:</b> {escape(result["prediction"])}
        </p>

        <p>
            <b>Confidence:</b>
            {result["confidence"][
                list(model.config.id2label.keys()).index(
                    int(next(
                        k for k, v in model.config.id2label.items()
                        if v == result["prediction"]
                    ))
                )] * 100:.2f}%
        </p>

        <h3>Tweet explanation</h3>

        <div style="font-size:20px; line-height:2;">
            {highlighted_tweet}
        </div>

        <hr>

        <p>
            <span style="background-color:rgba(0,180,0,0.5)">
            Positive contribution
            </span>

            <span style="background-color:rgba(220,0,0,0.5)">
            Negative contribution
            </span>
        </p>

    </body>
    </html>
    """

    with open("finbert_explanation.html", "w", encoding="utf-8") as f:
        f.write(html_content)

    print("Visualization saved as finbert_explanation.html")

In [27]:
result = explain_tweet(
    df_bear_but_pred_bull["text"].iloc[9]
)

visualize_tweet(result)

Tweet: Trans Mountain expansion costs soars to C$12.6B - report
Prediction: positive
Confidence: 0.9197039008140564

Token contributions:
[CLS]                +0.00000
trans                -0.03988
mountain             +0.02905
expansion            +0.48567
costs                +0.20672
so                   +0.11384
##ars                +0.21266
to                   +0.20806
c                    +0.06054
$                    +0.34086
12                   -0.00040
.                    +0.02922
6                    +0.33847
##b                  +0.09801
-                    +0.19800
report               -0.21585
[SEP]                +0.00000

Convergence delta: -0.13283610343933105
Visualization saved as finbert_explanation.html
